# Pose Preprocessing and PKL Creation

In this notebook, the extracted pose keypoints stored as `.npy` files are converted into the annotation format required by MMAction2.

## What is done

- Load pose `.npy` files
- Check file integrity and shapes
- Create train/val split information
- Build annotation dictionaries
- Save the final dataset as `rwf_pose.pkl`
- Verify the saved file

## Output

Saved annotation file:
data/processed/pose/pkl/rwf_pose.pkl

Setup + Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import numpy as np
import pickle
from tqdm import tqdm

BASE = "/content/drive/MyDrive/violence-detection"
POSE_ROOT = f"{BASE}/data/processed/pose/npy"
OUT_DIR = f"{BASE}/data/processed/pose/pkl"

os.makedirs(OUT_DIR, exist_ok=True)

print("POSE_ROOT:", POSE_ROOT)
print("OUT_DIR:", OUT_DIR)

Mounted at /content/drive
POSE_ROOT: /content/drive/MyDrive/violence-detection/data/processed/pose/npy
OUT_DIR: /content/drive/MyDrive/violence-detection/data/processed/pose/pkl


Optional sanity check on one .npy

In [ ]:
sample_path = sorted(glob.glob(os.path.join(POSE_ROOT, "train", "Fight", "*.npy")))[0]
sample_arr = np.load(sample_path)

print("Sample path:", sample_path)
print("Shape:", sample_arr.shape)
print("Dtype:", sample_arr.dtype)
print("Min:", sample_arr.min(), "Max:", sample_arr.max())

Sample path: /content/drive/MyDrive/violence-detection/data/processed/pose/npy/train/Fight/000000.npy
Shape: (150, 2, 17, 3)
Dtype: float32
Min: 0.0 Max: 277.99194


Build annotations and split dict

In [ ]:
annotations = []
split_dict = {
    "train": [],
    "val": []
}
label_map = {
    "NonFight": 0,
    "Fight": 1
}
bad_files = []

for split_name in ["train", "val"]:
    for class_name in ["Fight", "NonFight"]:
        folder = os.path.join(POSE_ROOT, split_name, class_name)
        npy_files = sorted(glob.glob(os.path.join(folder, "*.npy")))

        print(f"\nProcessing {split_name}/{class_name} -> {len(npy_files)} files")

        for npy_path in tqdm(npy_files):
            video_name = os.path.splitext(os.path.basename(npy_path))[0]
            frame_dir = f"{split_name}/{class_name}/{video_name}"

            try:
                arr = np.load(npy_path)
            except Exception as e:
                print(f"\nSkipping bad file: {npy_path}")
                print("Error:", e)
                bad_files.append((npy_path, str(e)))
                continue

            if arr.ndim != 4 or arr.shape[1:] != (2, 17, 3):
                print(f"\nSkipping unexpected shape: {npy_path} -> {arr.shape}")
                bad_files.append((npy_path, f"unexpected shape {arr.shape}"))
                continue

            T = arr.shape[0]
            coords = arr[..., :2]
            scores = arr[..., 2]
            keypoint = np.transpose(coords, (1, 0, 2, 3)).astype(np.float32)
            keypoint_score = np.transpose(scores, (1, 0, 2)).astype(np.float32)

            ann = {
                "frame_dir": frame_dir,
                "label": label_map[class_name],
                "total_frames": T,
                "keypoint": keypoint,
                "keypoint_score": keypoint_score
            }

            annotations.append(ann)
            split_dict[split_name].append(frame_dir)

print("\nFinished building annotations in memory.")
print("Total annotations:", len(annotations))
print("Train samples:", len(split_dict["train"]))
print("Val samples:", len(split_dict["val"]))
overlap = set(split_dict["train"]) & set(split_dict["val"])
print("Overlap:", len(overlap))
print("Bad files:", len(bad_files))


Processing train/Fight -> 800 files


 24%|██▍       | 191/800 [00:14<00:09, 61.68it/s]


Skipping bad file: /content/drive/MyDrive/violence-detection/data/processed/pose/npy/train/Fight/000142.npy
Error: No data left in file


100%|██████████| 800/800 [00:16<00:00, 48.24it/s] 



Processing train/NonFight -> 800 files


...


Processing val/Fight -> 200 files


100%|██████████| 200/200 [00:03<00:00, 59.72it/s] 



Processing val/NonFight -> 200 files


100%|██████████| 200/200 [00:03<00:00, 56.83it/s] 


Finished building annotations in memory.
Total annotations: 1999
Train samples: 1599
Val samples: 400
Overlap: 0
Bad files: 1


Save .pkl

In [ ]:
ann_file = os.path.join(OUT_DIR, "rwf_pose.pkl")
print("Annotation file path:", ann_file)

data = {
    "split": split_dict,
    "annotations": annotations
}

with open(ann_file, "wb") as f:
    pickle.dump(data, f)

print("Saved annotation file to:", ann_file)

Annotation file path: /content/drive/MyDrive/violence-detection/data/processed/pose/pkl/rwf_pose.pkl
Saved annotation file to: /content/drive/MyDrive/violence-detection/data/processed/pose/pkl/rwf_pose.pkl


Verify saved .pkl

In [ ]:
with open(ann_file, "rb") as f:
    data = pickle.load(f)

print(type(data))
print(data.keys())
print("Train split:", len(data["split"]["train"]))
print("Val split:", len(data["split"]["val"]))
print("Total annotations:", len(data["annotations"]))

sample = data["annotations"][0]

print("\nSample:")
print("frame_dir:", sample["frame_dir"])
print("label:", sample["label"])
print("total_frames:", sample["total_frames"])
print("keypoint shape:", sample["keypoint"].shape)
print("keypoint_score shape:", sample["keypoint_score"].shape)

<class 'dict'>
dict_keys(['split', 'annotations'])
Train split: 1599
Val split: 400
Total annotations: 1999

Sample:
frame_dir: train/Fight/000000
label: 1
total_frames: 150
keypoint shape: (2, 150, 17, 2)
keypoint_score shape: (2, 150, 17)
